In [1]:
import os
os.chdir('/workspace/fbe9ef30-c6cc-4cdd-bd0b-50ac68e6d8ac')
print(os.listdir('.'))


['.prompts', '-PROMPT-v6-DATASET.md', 'ldh_zeros_N5000_dps50.npy', '.kernel_llm_logs_1.txt', '.config', 'memory', 'ldh_worker.py']


In [2]:
import numpy as np
arr = np.load('ldh_zeros_N5000_dps50.npy')
print(arr.dtype, arr.shape)
print(arr[0], arr[-1])
print(arr[4999])


<U51 (5000,)
5.0941598445710949256987955170797974750670744531091 4981.1674898625791053105083724292317970900546075558
4981.1674898625791053105083724292317970900546075558


In [3]:
import mpmath as mp

# Build L_DH(s) — extract from worker (not just Z), so we can evaluate off-line.
def make_LDH(dps):
 mp.mp.dps = dps
 sqrt5 = mp.sqrt(5)
 ttl = (mp.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
 one_fifth = mp.mpf(1)/5
 two_fifth = mp.mpf(2)/5
 three_fifth = mp.mpf(3)/5
 four_fifth = mp.mpf(4)/5
 def L(s):
 s = mp.mpc(s)
 return mp.power(5, -s) * (
 mp.zeta(s, one_fifth)
 + ttl * mp.zeta(s, two_fifth)
 - ttl * mp.zeta(s, three_fifth)
 - mp.zeta(s, four_fifth)
 )
 return L

# Verify R2: known off-line zero approximate locations
mp.mp.dps = 30
L = make_LDH(30)
coords = [(0.808517, 85.699348),
 (0.650786, 114.163343),
 (0.574355, 166.479306),
 (0.724258, 176.702461)]
for sig, t in coords:
 v = L(mp.mpc(sig, t))
 print(f"({sig},{t}): |L| = {abs(v)}")


(0.808517,85.699348): |L| = 0.00000065115630697604509274686537049
(0.650786,114.163343): |L| = 0.0000403336866906579675344084026008
(0.574355,166.479306): |L| = 0.000000681549620376416260750359343042
(0.724258,176.702461): |L| = 0.000000470754691875852913289838743516


In [4]:
# Refine the four known off-line zeros using mp.findroot in 2D
mp.mp.dps = 50
L50 = make_LDH(50)
for sig, t in coords:
 s0 = mp.mpc(sig, t)
 try:
 r = mp.findroot(L50, s0, tol=mp.mpf(10)**-40)
 print(f"start ({sig},{t}) -> zero {r}, |L|={abs(L50(r))}")
 except Exception as e:
 print(f"failed at ({sig},{t}): {e}")


start (0.808517,85.699348) -> zero (0.80851718245663738555335196060684412785067026830502 + 85.699348485377592171929267708941729037987829423408j), |L|=4.4187072013762510949915625548031766561553398207193e-50


start (0.650786,114.163343) -> zero (0.65083008060973708240376064248770173385590565643697 + 114.16334273075698090416436263335003109324532818212j), |L|=8.0933469909195946941392992670499184086317122971493e-51


start (0.574355,166.479306) -> zero (0.574356050450805990721482105321702458531689961875 + 166.47930591316815587647765105775848526822008024556j), |L|=8.0015416332863919387629979644324691478061399810424e-50


start (0.724258,176.702461) -> zero (0.72425769462680978021118609148726731433857665933281 + 176.70246124285582505447387389275403813051469438992j), |L|=1.7191705119406800466101086215452163722655345690572e-49


In [5]:
# So we have a 2D search problem. Strategy: 
# For each grid line σ = const (σ in (0.5, 1.0)), scan t in [0,5000]. Look for sign changes
# in Re(L) and Im(L). Where both are near zero or where their isocurves cross,
# we have a candidate zero. We use that to find sign-change boxes.
# Note: by functional equation, off-line zeros come in symmetric pairs σ and 1-σ.
# So we only need to search σ in (0.5, 1.0] (and reflect).
# 
# Literature on DH function: known to have a small number of off-line zeros at low height.
# Reported zeros (Spira, Conrey-Ghosh, et al.) up to t~200 are typically the four listed.
# Let me look for more up to t=5000.
#
# Methodology: 2D grid scan. For each σ in a coarse grid, scan t finely. Identify candidate
# regions where |L| dips low.

import numpy as np
import mpmath as mp
from ldh_worker import find_zeros_in_block_v2 # not used here directly

# Quick check: at the critical line, the known critical zero density tells us how many zeros to expect.
# We have 5000 critical-line zeros up to t≈4981. Off-line zeros are rare for DH.

# Set up a moderate scan. Use dps=30 for scanning, then refine at dps=50.
print("Setting up scan...")


Setting up scan...


In [6]:
# Plan refinement.
# Idea: scan along VERTICAL lines σ = σ_k for σ_k ∈ {0.55, 0.60, 0.65, ..., 0.95} (or finer).
# On each vertical line, track Re(L(σ+it)) and Im(L(σ+it)) as functions of t.
# A zero is at a t-location where Re=Im=0. Equivalent: |L|^2 attains a local minimum near 0.
# So: detect local minima of |L(σ+it)| along each vertical line.
# Tag any local minimum where |L| < some threshold (say 0.01) as a candidate.
# Then use 2D findroot to refine.
# 
# This is a "minimum-following" strategy. The number of σ slices and t-step matters.
# 
# We saw all 4 known zeros have σ in [0.57,0.81], t up to 177. Let's pick σ slices 
# 0.52, 0.55, 0.58, 0.61, ..., 0.97 = step 0.03 → 16 slices.
# 
# Time check: how fast is L at dps=15 or 20 (mpmath)? Test:

import time
mp.mp.dps = 20
L = make_LDH(20)
t0 = time.time()
N=200
ts = np.linspace(100,110,N)
vals = [abs(L(mp.mpc(0.7, t))) for t in ts]
print("dps=20, %d evals in %.3fs (%.1f/s)" % (N, time.time()-t0, N/(time.time()-t0)))


dps=20, 200 evals in 2.822s (70.9/s)


In [7]:
# 70/s. For each σ slice with t∈[0,5000] at step Δt=0.05, that's 100k evals → 1400s.
# 16 slices → 22400s. Too slow.
# 
# Better strategy: leverage what we know.
# Known facts about DH:
# - It's a Hardy-Z-like function for the critical line. It has known off-line zeros at 
# low heights. Beyond a certain T, no off-line zeros have been found (and DH has been
# studied with rigor up to large T).
# - Specifically: Bombieri-Hejhal-Spira et al found ~37 off-line zeros up to t around 9000?
# Actually need to check. But many literature counts exist.
# 
# Let me use multiprocessing to parallelize. Check CPU count:

import os
print("CPUs:", os.cpu_count())


CPUs: 8


In [8]:
# With 8 CPUs and multiprocessing, we'd cut time to ~22400/8 ≈ 2800s ≈ 47 min. Feasible.
# But still let's reduce work intelligently.
# 
# IMPORTANT optimization: when scanning vertical line σ=const, we can use a coarser t-step 
# to detect dips, then refine. The off-line zeros are isolated points; |L|->small only near them.
# 
# A coarse step of Δt=0.5 captures dips in |L|. Then we refine around any candidate dip.
# 
# Step 1: For each σ_k in coarse σ-grid, evaluate |L| on coarse t-grid, find local minima.
# Step 2: refine around minima with finer t-grid, then 2D findroot.
# 
# Let's try step=0.5 first. Time per slice: 10000 evals → 140s. 16 slices → 2240s.
# With 8-way parallelism → ~280s. 

# Actually, even better: at LARGE σ (>~0.9), L is dominated by the n^{-s} sum which is well-behaved.
# At small σ (>0.5 close), behavior is more chaotic. Let's keep all.

# Let's prototype on a single σ slice first.
import numpy as np
import mpmath as mp
import time

def scan_sigma_slice(sigma, t_lo, t_hi, dt_scan, dps_scan):
 mp.mp.dps = dps_scan
 sqrt5 = mp.sqrt(5)
 ttl = (mp.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
 one_fifth = mp.mpf(1)/5
 two_fifth = mp.mpf(2)/5
 three_fifth = mp.mpf(3)/5
 four_fifth = mp.mpf(4)/5
 sig = mp.mpf(sigma)
 def L(t):
 s = mp.mpc(sig, t)
 return mp.power(5, -s) * (
 mp.zeta(s, one_fifth)
 + ttl * mp.zeta(s, two_fifth)
 - ttl * mp.zeta(s, three_fifth)
 - mp.zeta(s, four_fifth)
 )
 ts = []
 vs = []
 t = mp.mpf(t_lo)
 t_hi = mp.mpf(t_hi)
 step = mp.mpf(dt_scan)
 while t <= t_hi:
 ts.append(float(t))
 vs.append(float(abs(L(t))))
 t += step
 return np.array(ts), np.array(vs)

t0 = time.time()
ts, vs = scan_sigma_slice(0.7, 80, 180, 0.5, 20)
print(f"100 t-range in {time.time()-t0:.2f}s; min |L| = {vs.min()} at t={ts[vs.argmin()]}")


100 t-range in 4.08s; min |L| = 0.11370865141766795 at t=166.5


In [9]:
# 200 evals took 4s → 50/s, slightly slower than before. dt=0.5 over [0,5000] = 10000 evals → 200s per slice.
# That's 200s/CPU. With 8 CPUs in parallel, can do 8 slices in 200s. Total slices wanted: ~16-20.
# 
# Let's set up multiprocessing for slices.
# 
# But wait - we should think harder about detection. At σ=0.7, the dip at known off-line zero is only |L|≈0.11 at dt=0.5 resolution. Threshold 0.2 or 0.3 needed; this should still be feasible.
# 
# However, we need to be more careful. |L(s)| is bounded above and has typical magnitude that grows with t. Let me check the typical magnitude:

mp.mp.dps = 20
L = make_LDH(20)
for sigma in [0.55, 0.7, 0.85]:
 vs=[]
 for t in np.linspace(100,200,21):
 vs.append(float(abs(L(mp.mpc(sigma,t)))))
 print(f"sigma={sigma}: typical |L| ~ median={np.median(vs):.3f}, min={min(vs):.3f}, max={max(vs):.3f}")


sigma=0.55: typical |L| ~ median=0.981, min=0.166, max=3.762


sigma=0.7: typical |L| ~ median=0.819, min=0.366, max=2.769


sigma=0.85: typical |L| ~ median=0.888, min=0.422, max=2.178


In [10]:
# Typical |L| ~ 1. Dips at known off-line zeros are 0.11 ish at coarse step 0.5.
# Use threshold 0.25 to find candidate t-locations (local minima with |L| < 0.25). 
# Then refine in 2D.
# 
# But wait - the critical-line zeros at σ=0.5 will also create dips even at σ=0.55!
# Let me check that at σ=0.55, what happens near a critical-line zero (say t≈14.13):

mp.mp.dps = 20
L = make_LDH(20)
# pick one of the critical-line zeros, e.g. arr[0]=5.094...
gam = float(arr[0])
for s_re in [0.5, 0.51, 0.52, 0.55, 0.6, 0.7]:
 print(f"σ={s_re}, t={gam}: |L|={abs(L(mp.mpc(s_re, gam))):.4f}")


TypeError: unsupported format string passed to mpf.__format__

In [11]:
for s_re in [0.5, 0.51, 0.52, 0.55, 0.6, 0.7]:
 v = abs(L(mp.mpc(s_re, gam)))
 print(f"σ={s_re}, t={gam}: |L|={float(v):.4f}")


σ=0.5, t=5.094159844571095: |L|=0.0000
σ=0.51, t=5.094159844571095: |L|=0.0131
σ=0.52, t=5.094159844571095: |L|=0.0259
σ=0.55, t=5.094159844571095: |L|=0.0635
σ=0.6, t=5.094159844571095: |L|=0.1227
σ=0.7, t=5.094159844571095: |L|=0.2295


In [12]:
# So at σ=0.55, |L| is only 0.06 near a critical-line zero (which is NOT off-line). 
# Critical-line zeros bleed into σ slightly > 0.5.
# Strategy: skip σ-slices too close to 0.5, OR also check that the resulting 2D-refined zero
# is NOT on the line.
# 
# Cleaner: Run findroot from each dip candidate. If it converges to a complex zero with 
# Re(s) > 0.5 + ε (say ε=0.01), accept; else reject as critical-line zero.
# 
# Plan:
# 1. For σ_k ∈ {0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95}, scan t∈[0,5000] at dt=0.5, dps=20.
# 2. Find local minima with |L(σ_k + it)| < 0.30.
# 3. For each candidate (σ_k, t_min), call mp.findroot starting at s0=σ_k + i*t_min at dps=50.
# 4. Accept zero if Re(s_zero) > 0.5 + 0.005 AND |L(s_zero)| < 1e-40.
# 5. Deduplicate (collect zeros with σ>0.5; their mirrors at 1-σ are automatic).
#
# Use multiprocessing.Pool. But the worker needs to be pickle-safe (module-level fn).
# 
# Save worker to disk:

worker_code = '''
import mpmath as mp
import numpy as np

def make_L(dps):
 mp.mp.dps = dps
 sqrt5 = mp.sqrt(5)
 ttl = (mp.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
 one_fifth = mp.mpf(1)/5
 two_fifth = mp.mpf(2)/5
 three_fifth = mp.mpf(3)/5
 four_fifth = mp.mpf(4)/5
 def L(s):
 s = mp.mpc(s)
 return mp.power(5, -s) * (
 mp.zeta(s, one_fifth)
 + ttl * mp.zeta(s, two_fifth)
 - ttl * mp.zeta(s, three_fifth)
 - mp.zeta(s, four_fifth)
 )
 return L

def scan_one_sigma(args):
 """Scan a vertical line σ=const, return list of (sigma, t) candidate dips."""
 sigma, t_lo, t_hi, dt_scan, dps_scan, threshold = args
 mp.mp.dps = dps_scan
 L = make_L(dps_scan)
 sig = mp.mpf(sigma)
 # Build t-grid and evaluate |L|
 ts = []
 vs = []
 t = mp.mpf(t_lo)
 step = mp.mpf(dt_scan)
 thi = mp.mpf(t_hi)
 while t <= thi:
 ts.append(float(t))
 vs.append(float(abs(L(mp.mpc(sig, t)))))
 t += step
 ts = np.array(ts); vs = np.array(vs)
 # Find local minima
 candidates = []
 for i in range(1, len(vs)-1):
 if vs[i] < vs[i-1] and vs[i] < vs[i+1] and vs[i] < threshold:
 candidates.append((float(sigma), float(ts[i]), float(vs[i])))
 return candidates
'''
with open('ldh_2d_worker.py','w') as f:
 f.write(worker_code)
print("written")


written


In [13]:
# Quick sanity check: scan one σ slice over short range and detect known zero.
from importlib import reload
import ldh_2d_worker as w
reload(w)
import time
t0 = time.time()
cands = w.scan_one_sigma((0.7, 80, 200, 0.5, 20, 0.30))
print(f"In {time.time()-t0:.1f}s, found candidates:")
for c in cands: print(c)


In 4.8s, found candidates:
(0.7, 114.0, 0.1716477346711215)
(0.7, 131.5, 0.288963486746368)
(0.7, 149.5, 0.2719031510999511)
(0.7, 166.5, 0.11370865141766795)
(0.7, 176.5, 0.27945822750599064)


In [14]:
# Good. 5 candidates: 2 known off-line zeros at 114.16 and 166.48 and 176.70 + spurious ones.
# 
# Note 176.70 was hit at threshold 0.279 (close to 0.30). To be safe let's use 0.30 or 0.35.
# But more spurious means more findroot work. Reasonable.
# 
# Now estimate full scan cost. dt=0.5, t∈[0,5000] = 10000 evals per slice. At ~50/s → 200s/slice.
# 9 slices in [0.55, 0.95] step 0.05 → ~1800s sequential, ~225s parallel with 8 cores.
# 
# Let's start small: σ_k = [0.52, 0.55, 0.58, 0.62, 0.65, 0.68, 0.72, 0.75, 0.78, 0.82, 0.85, 0.88, 0.92, 0.95, 0.98]
# That's 15 slices. ~3000s sequential, ~400s parallel.
# 
# Actually, let me first test if σ_k near 0.5 will be polluted by critical-line zeros.
# At σ=0.55, near critical-line zero gam, |L|~0.06. With 5000 critical zeros in [0,5000], 
# all will produce dips → enormous spurious list. So skip σ_k < 0.55 to be safe, or use 
# higher threshold near these.
# 
# Better: skip σ_k = 0.52, 0.55, 0.58. Use σ_k starting at 0.60. Most known off-line zeros 
# have Re > 0.57; we'd miss zeros with Re ∈ [0.51, 0.59]. 
# 
# Alternative: at σ=0.55 use a STRICTER threshold like |L|<0.03. The known off-line zero at 
# σ=0.5745 produces what at σ=0.55? Let's check:

mp.mp.dps = 20
L = make_LDH(20)
# Off-line zero at (0.574355, 166.479306)
for sig in [0.52, 0.54, 0.55, 0.56, 0.58, 0.60]:
 v = abs(L(mp.mpc(sig, 166.48)))
 print(f"σ={sig}: |L|={float(v):.4f}")


σ=0.52: |L|=0.0254
σ=0.54: |L|=0.0185
σ=0.55: |L|=0.0140
σ=0.56: |L|=0.0087
σ=0.58: |L|=0.0038
σ=0.6: |L|=0.0183


In [15]:
# So at σ=0.58 we are very close to the (0.5744, 166.48) zero. Good.
# 
# Let me think differently: instead of mostly relying on σ slices, I'll use a 2D grid of σ-points and run 
# vertical scans only at σ values where it's safe to assume critical-line bleed isn't dominant.
# 
# OPTIMAL: Since we have the 5000 critical-line zeros cached, we KNOW where they are. 
# Most σ-slices at fixed σ>0.5 will have dips near every critical-line zero spaced by their gaps (~2pi/log).
# These dips fade as σ grows. By σ=0.6, do they fade enough?

# Let me test extensively at σ=0.6, looking near several critical zeros:
mp.mp.dps = 20
L = make_LDH(20)
gams = [float(arr[i]) for i in [0, 10, 50, 100]]
print("σ=0.6 |L| at critical-line zero t-values:")
for g in gams:
 print(f" t={g}: |L|={float(abs(L(mp.mpc(0.6,g)))):.4f}")
print("σ=0.6 |L| midway between critical zeros (typical):")
for i in [5, 25, 75]:
 g = (float(arr[i]) + float(arr[i+1]))/2
 print(f" t={g}: |L|={float(abs(L(mp.mpc(0.6,g)))):.4f}")


σ=0.6 |L| at critical-line zero t-values:
 t=5.094159844571095: |L|=0.1227
 t=30.159417906273386: |L|=0.2678
 t=97.973624940697: |L|=0.4225
 t=171.04054626516495: |L|=0.3859
σ=0.6 |L| midway between critical zeros (typical):
 t=20.734253969661022: |L|=2.4146
 t=57.71809981577866: |L|=1.1085
 t=136.15786459156269: |L|=1.8619


In [16]:
# Hmm, at σ=0.6 even t=5 (near first crit zero) has |L|=0.12 — would pass threshold 0.30.
# So critical-line bleed is still significant at σ=0.6.
# 
# Strategy revision: For each σ-slice, ANY dip is a candidate, but in refining we'll just 
# do findroot and check Re(zero) > 0.5+ε.
# Each candidate runs in ~1s of findroot. With ~5000 critical-line zeros plus a few off-line, 
# we'd have ~5000 candidates per σ slice. Refinement cost: 5000 candidates × 1s × 9 slices = 12.5h. Too slow.
# 
# Better: at lower σ (0.55-0.70), critical-line dips DOMINATE. Skip those σ slices entirely.
# At higher σ (>0.75 or so), critical-line dips fade enough to be filtered by tighter threshold.
# 
# Actually let me think more carefully. What we want:
# - Detect off-line zeros at (σ_0, t_0) with σ_0 between 0.5 and 1.0.
# - The function L near (σ_0, t_0) has |L| approaching 0.
# - Critical-line zero at (0.5, γ_n) makes |L(σ_0+iγ_n)| ≈ const·(σ_0-0.5) for small σ_0-0.5,
# so at σ=0.7, |L|≈0.2 near γ_n. At σ=1.0, |L|≈0.5 typically.
# 
# Idea: Run scan at σ = 0.7, 0.8, 0.9. At these σ, critical-line bleed is moderate (|L|≈0.2-0.5 
# at γ_n locations), so set threshold tight, e.g. 0.05. But would we still detect known off-line zeros?

mp.mp.dps = 20
L = make_LDH(20)
# Off-line zeros at (sig, t). Let's check |L(0.7+it)| at t = known off-line t:
for sig, t in coords:
 v = float(abs(L(mp.mpc(0.7, t))))
 print(f"σ=0.7, off-line t={t}: |L|={v:.4f}")
print()
# And |L(0.8+it)|:
for sig, t in coords:
 v = float(abs(L(mp.mpc(0.8, t))))
 print(f"σ=0.8, off-line t={t}: |L|={v:.4f}")


σ=0.7, off-line t=85.699348: |L|=0.1380
σ=0.7, off-line t=114.163343: |L|=0.0472
σ=0.7, off-line t=166.479306: |L|=0.1124
σ=0.7, off-line t=176.702461: |L|=0.0292

σ=0.8, off-line t=85.699348: |L|=0.0107
σ=0.8, off-line t=114.163343: |L|=0.1506
σ=0.8, off-line t=166.479306: |L|=0.2216
σ=0.8, off-line t=176.702461: |L|=0.0908


In [17]:
# Crucial observation: An off-line zero at (σ_0, t_0) creates a dip on σ=σ_k line near t_0 
# of depth ~ |σ_k - σ_0|. So we MUST have σ_k close enough to σ_0 to make it stand out.
# 
# The four known off-line zeros span σ ∈ [0.57, 0.81]. With σ_k step of 0.05, the maximum 
# distance is 0.025, giving dip depth ~0.05 → comparable to critical-line bleed at σ=0.6-0.7.
# 
# This requires FINER σ-grid. Let me use σ_k step 0.025 from 0.525 to 0.975. That's 19 slices.
# 
# Alternative approach (much better): The functional equation: at σ_0=0.5+a, mirror is at 1-σ_0=0.5-a.
# So if there's an off-line zero at σ_0>0.5, its mirror is at σ_0<0.5. We can detect on BOTH sides.
# 
# But the real winning approach: USE A 2D NEWTON METHOD. From any starting point in a dip 
# region, mp.findroot converges quadratically. So pick starting points where |L| is locally small.
# 
# I'll use ARGUMENT PRINCIPLE-style approach: scan along a single vertical line σ=σ_0 close to 
# critical line at, say, σ_0=0.6, finding ALL local minima (including those due to critical-line 
# zeros and off-line zeros). For each minimum at (0.6, t_min), do 2D findroot starting at (0.6, t_min).
# The Newton iteration will converge to the nearest zero — for critical-line zeros, it goes to (0.5, γ),
# for off-line, it goes to (σ_off, t_off). Then we filter by Re(zero)>0.5+ε.
# 
# Problem: findroot might find the wrong zero or fail to converge. Let me test:

mp.mp.dps = 30
L30 = make_LDH(30)
# Start near (0.6, 5.094) (critical-line zero at t=5.094)
try:
 r = mp.findroot(L30, mp.mpc(0.6, 5.094), tol=mp.mpf(10)**-25)
 print(f"From (0.6,5.094) -> {r}, |L|={float(abs(L30(r))):.2e}")
except Exception as e:
 print(f"Fail: {e}")

# Start near (0.6, 114.16) — known off-line at (0.65083, 114.163)
try:
 r = mp.findroot(L30, mp.mpc(0.6, 114.16), tol=mp.mpf(10)**-25)
 print(f"From (0.6,114.16) -> {r}, |L|={float(abs(L30(r))):.2e}")
except Exception as e:
 print(f"Fail: {e}")


From (0.6,5.094) -> (0.5 + 5.09415984457109492569879551708j), |L|=1.33e-31


From (0.6,114.16) -> (0.650830080609737082403760642489 + 114.163342730756980904164362633j), |L|=4.38e-30


In [18]:
# Excellent! Newton finds the right zero — converging to (0.5, 5.094) for critical-line near a 
# critical zero, and to off-line zero from (0.6, 114.16). 
# So the plan: scan vertical line σ=0.6 (close enough to critical line that all critical zeros are
# captured at fairly low threshold AND off-line zeros up to σ≈0.8 are also captured at threshold),
# then findroot all dips, filter by Re(zero) > 0.5 + 0.005.
# 
# Wait — but to detect off-line zero at σ=0.8 from σ=0.6 line, dip depth would be |σ-0.6|^? = ?
# The zero at (0.81, 85.7): how does |L(0.6+it)| look near t=85.7? Let me check:

mp.mp.dps = 20
L = make_LDH(20)
print("σ=0.6, near (0.8085, 85.6993):")
for t in np.linspace(84, 87, 13):
 v = float(abs(L(mp.mpc(0.6, t))))
 print(f" t={t:.2f}: |L|={v:.3f}")


σ=0.6, near (0.8085, 85.6993):
 t=84.00: |L|=3.210
 t=84.25: |L|=3.368
 t=84.50: |L|=3.074
 t=84.75: |L|=2.435
 t=85.00: |L|=1.641
 t=85.25: |L|=0.905
 t=85.50: |L|=0.406
 t=85.75: |L|=0.267
 t=86.00: |L|=0.519
 t=86.25: |L|=0.985
 t=86.50: |L|=1.480
 t=86.75: |L|=1.806
 t=87.00: |L|=1.801


In [19]:
# Good — there's a clean dip down to 0.27 at t≈85.75 even from σ=0.6, well-distinguished. ✓
# 
# So strategy: SCAN ONLY σ=0.6 line with threshold 0.30. All zeros (critical-line + off-line) 
# produce dips. Then findroot from each candidate; filter.
# 
# Cost: 10000 evals at dps=20 (~50/s) = 200s. ~5000 candidates from critical-line zeros + maybe 50 
# extra from off-line. Findroot: ~1s × 5050 = ~5050s = 1.4h. With parallelism → 10 min.
#
# Even better idea: Since I have the critical zero list, I can SKIP redoing them. Just scan, 
# find candidates, and for each candidate check if it's near a known critical zero (within some 
# t-tolerance, say 0.5). If yes, skip findroot. If no, run findroot — it's a potential off-line.
#
# But that's not perfectly safe. Let me do it more cleanly: just findroot every candidate; we'll 
# parallelize.

# Better still: instead of scanning ONLY σ=0.6, also try σ=0.7, 0.8, 0.9. The redundancy ensures 
# we find off-line zeros (their dips appear strongest on σ near σ_off). And critical-line bleed 
# decreases at higher σ, reducing spurious candidates.
# 
# Actually I realize: for each off-line zero (σ_off, t_off), we want at least one σ slice within 
# 0.05 of σ_off so the dip is deep. With σ-grid {0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 
# 0.95}, every σ_off has a slice within 0.025. Good.
# 
# But this leads to LOTS of findroots due to critical-line bleed.
# 
# RESOLVED PLAN: 
# Scan along σ_k ∈ {0.65, 0.75, 0.85, 0.95}. Step σ=0.10. Use tight threshold (0.10) so we 
# skip most critical-line dips at high σ. 
# This misses off-line zeros with σ_off ∈ (0.6, 0.65) gap. Hmm.
# 
# Let me just go with comprehensive: σ_k ∈ {0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95}, 
# threshold 0.10. Critical-line bleed at σ=0.55, 0.6 will be huge problem. Skip them.
# 
# Plan v3: σ_k ∈ {0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95}, threshold 0.15.
# 
# Sanity: off-line zero at (0.57, 166.5) → from σ=0.65 line, dip depth |L(0.65, 166.5)| = ?

mp.mp.dps = 20
L = make_LDH(20)
for sig_test in [0.55, 0.60, 0.65, 0.70, 0.75]:
 v = float(abs(L(mp.mpc(sig_test, 166.479))))
 print(f"σ={sig_test}, t=166.479 (zero is at σ=0.574): |L|={v:.4f}")


σ=0.55, t=166.479 (zero is at σ=0.574): |L|=0.0140
σ=0.6, t=166.479 (zero is at σ=0.574): |L|=0.0183
σ=0.65, t=166.479 (zero is at σ=0.574): |L|=0.0619
σ=0.7, t=166.479 (zero is at σ=0.574): |L|=0.1124
σ=0.75, t=166.479 (zero is at σ=0.574): |L|=0.1664


In [20]:
# From σ=0.65, dip is 0.062 — easily detectable with threshold 0.15.
# 
# Now estimate critical-line bleed at σ=0.65: |L(0.65 + iγ_n)| ranges from ~0.06 to ~0.5 (depends 
# on local Z-amplitude). So threshold 0.15 will catch many crit zeros too. Need to refine all.
# 
# Let me think differently. Do the σ=0.6 scan AND filter against the known critical-line zero 
# list! Skip candidates within ~0.5 of any γ_n.
# 
# Actually here's the cleanest version:
# 
# - For each σ_k ∈ {0.65, 0.75, 0.85, 0.95}, scan t∈[0,5000] at step 0.5 dps=20.
# - Find local minima with |L| < 0.20.
# - DEDUP each candidate against known critical-line γ_n: if |t_cand - γ_n| < 0.5, skip 
# (already accounted for by critical-line zero).
# - For remaining candidates, run findroot at dps=50.
# - If converged to zero with Re(zero) > 0.5+0.005 and |L|<1e-40 and t<5000, KEEP.
# - Dedup across σ_k slices.
#
# Let's check: σ=0.65 critical-line bleed. With γ_n's the closest critical zero to dip location, 
# if the dip is FROM a critical-line zero, then t_cand ≈ γ_n. Good, this dedup works.
# 
# But if a dip is from an off-line zero at (σ_off, t_off), t_cand ≈ t_off, which is NOT near 
# any γ_n in general. Good.
# 
# However, off-line zeros tend to come in pairs near critical-line zeros — wait, no. The DH 
# function may have off-line zeros that COINCIDE in t with a critical-line zero? Let me check the 
# four known:
# t-values of off-line: 85.699, 114.163, 166.479, 176.702
# Are any close to a critical-line zero?
import numpy as np
crit_t = np.array([float(x) for x in arr])
for sigma, t in coords:
 nearest_idx = np.argmin(np.abs(crit_t - t))
 print(f"off-line ({sigma}, {t}): nearest crit t = {crit_t[nearest_idx]:.4f}, dt={t-crit_t[nearest_idx]:+.4f}")


off-line (0.808517, 85.699348): nearest crit t = 87.6475, dt=-1.9481
off-line (0.650786, 114.163343): nearest crit t = 112.3764, dt=+1.7869
off-line (0.574355, 166.479306): nearest crit t = 168.3242, dt=-1.8449
off-line (0.724258, 176.702461): nearest crit t = 178.1699, dt=-1.4674


In [21]:
# All off-line zeros are ≥1.4 away from nearest critical-line zero. Safe to use a dedup window 
# of ~0.5. (We could even widen it slightly — but a window too wide misses off-line zeros that 
# happen to be near a critical zero. 0.5 is conservative.)
# 
# Now, let me also account for: off-line zeros at very low σ_0 (close to 0.5) — but those are 
# also detected from σ=0.65 if any. And they'll be filtered if too close to a crit zero. 
# 
# Hmm, what if there's an off-line zero at (0.51, t_0)? It would have a very small "off-ness" 
# and be hard to distinguish from a critical-line zero. Limitation noted.
# 
# OK let's execute. Run σ=0.65, 0.75, 0.85, 0.95 in parallel.

import multiprocessing as mp_proc
import time
import importlib
import ldh_2d_worker
importlib.reload(ldh_2d_worker)
from ldh_2d_worker import scan_one_sigma

def parallel_scan(sigmas, t_lo, t_hi, dt, dps, threshold, n_workers=8):
 args = [(s, t_lo, t_hi, dt, dps, threshold) for s in sigmas]
 with mp_proc.Pool(n_workers) as p:
 results = p.map(scan_one_sigma, args)
 return results

# Test small first
t0 = time.time()
results = parallel_scan([0.65, 0.75, 0.85, 0.95], 0, 200, 0.5, 20, 0.20, 4)
print(f"Test scan [0,200] in {time.time()-t0:.1f}s")
for s, r in zip([0.65, 0.75, 0.85, 0.95], results):
 print(f"σ={s}: {len(r)} candidates")
 for c in r: print(f" {c}")


Test scan [0,200] in 6.7s
σ=0.65: 3 candidates
 (0.65, 40.5, 0.177712323944778)
 (0.65, 114.0, 0.16128050776656846)
 (0.65, 166.5, 0.06418334769323623)
σ=0.75: 2 candidates
 (0.75, 114.0, 0.1943598043811149)
 (0.75, 166.5, 0.1671435591353544)
σ=0.85: 0 candidates
σ=0.95: 0 candidates


In [22]:
# σ=0.85: 0 candidates! That means the off-line zero at (0.808, 85.7) was missed at threshold 0.20.
# Need looser threshold OR finer σ-grid.
# Let me check |L(0.85, t)| near 85.7:

mp.mp.dps = 20
L = make_LDH(20)
for t in np.linspace(85.0, 86.0, 11):
 v = float(abs(L(mp.mpc(0.85, t))))
 print(f"σ=0.85, t={t:.2f}: |L|={v:.4f}")


σ=0.85, t=85.00: |L|=1.1585
σ=0.85, t=85.10: |L|=0.9559
σ=0.85, t=85.20: |L|=0.7618
σ=0.85, t=85.30: |L|=0.5809
σ=0.85, t=85.40: |L|=0.4160
σ=0.85, t=85.50: |L|=0.2680
σ=0.85, t=85.60: |L|=0.1367
σ=0.85, t=85.70: |L|=0.0515
σ=0.85, t=85.80: |L|=0.1342
σ=0.85, t=85.90: |L|=0.2538
σ=0.85, t=86.00: |L|=0.3814


In [23]:
# With dt=0.5, our grid samples at t=85.5 (|L|=0.27) and t=86.0 (|L|=0.38). The minimum is between them. Local min check requires consecutive: v[85.5] < v[85.0]=1.16 AND v[85.5] < v[86.0]=0.38. So 85.5 IS a local min with |L|=0.27, but our threshold was 0.20. Need looser threshold OR finer dt.
# 
# Let me use threshold 0.40 and dt=0.5. Then we'll have more candidates including critical-line bleed.
# Then after dedup against critical-line zeros, runs findroot on remaining.
# 
# OR finer dt = 0.25 + threshold 0.20. dt=0.25 doubles cost. Let me go threshold 0.40 dt=0.5.

t0 = time.time()
results = parallel_scan([0.65, 0.75, 0.85, 0.95], 0, 200, 0.5, 20, 0.40, 4)
print(f"Test scan [0,200] in {time.time()-t0:.1f}s")
for s, r in zip([0.65, 0.75, 0.85, 0.95], results):
 print(f"σ={s}: {len(r)} candidates")
 for c in r: print(f" {c}")


Test scan [0,200] in 6.5s
σ=0.65: 19 candidates
 (0.65, 5.0, 0.20760404730062446)
 (0.65, 9.0, 0.29318649289362325)
 (0.65, 12.0, 0.36447970543507013)
 (0.65, 14.5, 0.34493874828015164)
 (0.65, 22.0, 0.38824743270376216)
 (0.65, 23.5, 0.3926686368101319)
 (0.65, 32.0, 0.3414429417606321)
 (0.65, 40.5, 0.177712323944778)
 (0.65, 58.5, 0.397076588803342)
 (0.65, 68.5, 0.23295534114584854)
 (0.65, 85.5, 0.3564065952908154)
 (0.65, 103.5, 0.36395414130730885)
 (0.65, 114.0, 0.16128050776656846)
 (0.65, 131.5, 0.2372821527757412)
 (0.65, 149.5, 0.24310424888542168)
 (0.65, 159.5, 0.29918712503584255)
 (0.65, 166.5, 0.06418334769323623)
 (0.65, 176.5, 0.2995007801550383)
 (0.65, 195.0, 0.35483855461855174)
σ=0.75: 11 candidates
 (0.75, 5.0, 0.29333983387683)
 (0.75, 40.5, 0.22494753802696957)
 (0.75, 68.5, 0.28900567872236815)
 (0.75, 85.5, 0.28390590324483495)
 (0.75, 114.0, 0.1943598043811149)
 (0.75, 131.5, 0.33298925403819146)
 (0.75, 149.5, 0.3039121340656443)
 (0.75, 159.5, 0.381406977

In [24]:
# Great. With threshold 0.40 we capture 85.5 (off-line at 0.808) from all σ slices.
# Now let's check which t-values are near critical-line zeros (within 0.5):
import numpy as np
crit_t = np.array([float(x) for x in arr])
near_crit = []
candidates_all = []
for sigma_slice, cs in zip([0.65, 0.75, 0.85, 0.95], results):
 for sigma, t, v in cs:
 d_min = np.min(np.abs(crit_t - t))
 candidates_all.append((sigma, t, v, d_min))
 if d_min < 0.5:
 near_crit.append((sigma, t, v, d_min))
print("Candidates within 0.5 of crit zero (likely bleed):", len(near_crit))
for c in near_crit[:20]:
 print(c)
print("Candidates NOT near crit zero (off-line candidates):")
for c in candidates_all:
 if c[3] >= 0.5:
 print(c)


Candidates within 0.5 of crit zero (likely bleed): 27
(0.65, 5.0, 0.20760404730062446, np.float64(0.09415984457109516))
(0.65, 9.0, 0.29318649289362325, np.float64(0.0600855919034764))
(0.65, 12.0, 0.36447970543507013, np.float64(0.13354542580074202))
(0.65, 14.5, 0.34493874828015164, np.float64(0.09599688772250126))
(0.65, 22.0, 0.38824743270376216, np.float64(0.15970776509397666))
(0.65, 23.5, 0.3926686368101319, np.float64(0.15462988797322552))
(0.65, 32.0, 0.3414429417606321, np.float64(0.03550002883646286))
(0.65, 40.5, 0.177712323944778, np.float64(0.18295279706700285))
(0.65, 58.5, 0.397076588803342, np.float64(0.10489705466746813))
(0.65, 68.5, 0.23295534114584854, np.float64(0.15290052731202763))
(0.65, 103.5, 0.36395414130730885, np.float64(0.1474091376740745))
(0.65, 131.5, 0.2372821527757412, np.float64(0.018743993297619))
(0.65, 149.5, 0.24310424888542168, np.float64(0.14711715555313276))
(0.65, 159.5, 0.29918712503584255, np.float64(0.1007686246057915))
(0.65, 195.0, 0.35

In [25]:
# Beautiful. All 4 known off-line zeros are detected as candidates NOT near critical-line zeros.
# 16 such candidates (4 zeros × 4 σ slices). After dedup these are 4 unique zeros.
# 
# Now let's scale up to full range t ∈ [0, 5000]. Time estimate: 10000 evals × 4 slices = 40000 evals.
# Single slice ~200s (at ~50/s). With 4-way parallelism, ~200s wall time.
# 
# Let me run it. Use 4 slices × 8 workers (parallel) — only use 4 workers since we have 4 slices.

import time
t0 = time.time()
results_full = parallel_scan([0.65, 0.75, 0.85, 0.95], 0, 5000, 0.5, 20, 0.40, 4)
print(f"Full scan in {time.time()-t0:.1f}s")
for s, r in zip([0.65, 0.75, 0.85, 0.95], results_full):
 print(f"σ={s}: {len(r)} candidates")


TimeoutError: Code execution timed out after 900.0 seconds

In [26]:
# Timed out. Let me make it more incremental and faster.
# Save intermediate results to disk so we can pick up if interrupted.
# Run slices independently with shorter timeouts.
# 
# Estimate: 10000 mpc evals at dps=20 took us ~200s. But L involves 4 hurwitz zetas. Actually 
# the test of 200 evals in 4s gives 50/s, so 10000 evals = 200s. 4 slices in parallel with 4 workers
# should be 200s, but it timed out at 900s — workers might be slower or mpmath thread settings.
# 
# Let me split each slice into smaller t-chunks and parallelize within each slice across all 8 cores.
# Each chunk: t in [t_lo, t_hi], dt=0.5, dps=20.
# 8 chunks per slice × 4 slices = 32 chunks. ~625 evals/chunk → 12s/chunk. With 8 cores in parallel,
# 4 batches → 48s per slice. Total 4 slices = 200s wall. Let me try.

# Or just decrease t_hi and check progress.

# Plan: split full t range [0, 5000] into chunks of 250. 20 chunks per slice. With 8 cores, 
# 20*4 = 80 jobs, batched in 8 at a time = 10 batches × per-job time.
# Per-job time: 500 evals at 50/s = 10s. 10 batches × 10s = 100s wall. ✓
# 
# Let's implement.

# Update worker to do chunks:
worker_code = '''
import mpmath as mp
import numpy as np

def make_L(dps):
 mp.mp.dps = dps
 sqrt5 = mp.sqrt(5)
 ttl = (mp.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
 one_fifth = mp.mpf(1)/5
 two_fifth = mp.mpf(2)/5
 three_fifth = mp.mpf(3)/5
 four_fifth = mp.mpf(4)/5
 def L(s):
 s = mp.mpc(s)
 return mp.power(5, -s) * (
 mp.zeta(s, one_fifth)
 + ttl * mp.zeta(s, two_fifth)
 - ttl * mp.zeta(s, three_fifth)
 - mp.zeta(s, four_fifth)
 )
 return L

def scan_chunk(args):
 """Scan a vertical line σ=const over a t-chunk. Return raw |L| samples."""
 sigma, t_lo, t_hi, dt_scan, dps_scan = args
 mp.mp.dps = dps_scan
 L = make_L(dps_scan)
 sig = mp.mpf(sigma)
 ts = []
 vs = []
 t = mp.mpf(t_lo)
 step = mp.mpf(dt_scan)
 thi = mp.mpf(t_hi)
 while t <= thi:
 ts.append(float(t))
 vs.append(float(abs(L(mp.mpc(sig, t)))))
 t += step
 return (sigma, t_lo, t_hi, np.array(ts), np.array(vs))

def refine_candidate(args):
 """Run mp.findroot at dps_final from (sigma0, t0). Return zero or None."""
 sigma0, t0, dps_final, tol_exp = args
 mp.mp.dps = dps_final
 L = make_L(dps_final)
 s0 = mp.mpc(sigma0, t0)
 try:
 r = mp.findroot(L, s0, tol=mp.mpf(10)**(-tol_exp))
 v = abs(L(r))
 return (float(r.real), float(r.imag), str(r.real), str(r.imag), float(v))
 except Exception as e:
 return None
'''
with open('ldh_2d_worker.py','w') as f:
 f.write(worker_code)
print("written")


written


In [27]:
# Run scan in chunks of 100 t-units, fully parallel
import importlib, ldh_2d_worker
importlib.reload(ldh_2d_worker)
from ldh_2d_worker import scan_chunk
import multiprocessing as mp_proc
import numpy as np, time, pickle, os

SIGMAS = [0.65, 0.75, 0.85, 0.95]
T_HI = 5000
CHUNK = 250
DT = 0.5
DPS = 20

jobs = []
for s in SIGMAS:
 t = 0
 while t < T_HI:
 t_next = min(t + CHUNK, T_HI)
 jobs.append((s, t, t_next, DT, DPS))
 t = t_next
print(f"Total jobs: {len(jobs)}")
# Estimate
n_eval = sum((j[2]-j[1])/j[3]+1 for j in jobs)
print(f"Total evals: {n_eval:.0f}")


Total jobs: 80
Total evals: 40080


In [28]:
# Run in batches
import time
t0 = time.time()
all_chunks = []
with mp_proc.Pool(8) as p:
 # use imap_unordered to get results as they come in
 for i, res in enumerate(p.imap_unordered(scan_chunk, jobs)):
 all_chunks.append(res)
 if (i+1) % 10 == 0:
 print(f" {i+1}/{len(jobs)} chunks done in {time.time()-t0:.1f}s")
print(f"Total scan time: {time.time()-t0:.1f}s, {len(all_chunks)} chunks")


TimeoutError: Code execution timed out after 900.0 seconds

In [29]:

# Check what was produced
print(f"Chunks done so far: {len(all_chunks)}")
if all_chunks:
 # Pickle progress
 with open('partial_chunks.pkl','wb') as f:
 pickle.dump(all_chunks, f)
 print("Saved partial.")
 # Show coverage
 done_jobs = set((c[0], c[1], c[2]) for c in all_chunks)
 print(f"Unique done jobs: {len(done_jobs)}")


Chunks done so far: 48
Saved partial.
Unique done jobs: 48


In [30]:
# 48/80 done in 900s. We need ~600s more for remaining 32.
# Let me run remaining jobs.
remaining = [j for j in jobs if (j[0], j[1], j[2]) not in done_jobs]
print(f"Remaining jobs: {len(remaining)}")

t0 = time.time()
with mp_proc.Pool(8) as p:
 for i, res in enumerate(p.imap_unordered(scan_chunk, remaining)):
 all_chunks.append(res)
 if (i+1) % 5 == 0:
 print(f" {i+1}/{len(remaining)} new chunks in {time.time()-t0:.1f}s")
print(f"Done in {time.time()-t0:.1f}s")
with open('partial_chunks.pkl','wb') as f:
 pickle.dump(all_chunks, f)
print("Saved final.")


Remaining jobs: 32


 5/32 new chunks in 156.9s


 10/32 new chunks in 190.4s


 15/32 new chunks in 298.8s


 20/32 new chunks in 443.7s


 25/32 new chunks in 571.9s


 30/32 new chunks in 674.5s


Done in 687.8s
Saved final.


In [31]:
# Time is short. Consolidate chunk data, find candidates, dedup against critical zeros, 
# refine ONLY non-critical candidates with findroot.
import numpy as np, pickle, time

# Sort chunks by (sigma, t_lo)
all_chunks_sorted = sorted(all_chunks, key=lambda c: (c[0], c[1]))

# Concatenate per σ
slice_data = {}
for s in SIGMAS:
 cs = [c for c in all_chunks_sorted if c[0]==s]
 ts = np.concatenate([c[3] for c in cs])
 vs = np.concatenate([c[4] for c in cs])
 # sort & dedup (boundary points)
 order = np.argsort(ts)
 ts = ts[order]; vs = vs[order]
 # remove duplicates
 uniq_mask = np.concatenate([[True], np.diff(ts) > 1e-9])
 ts = ts[uniq_mask]; vs = vs[uniq_mask]
 slice_data[s] = (ts, vs)
 print(f"σ={s}: {len(ts)} samples, t range {ts[0]:.1f}-{ts[-1]:.1f}")


σ=0.65: 10001 samples, t range 0.0-5000.0
σ=0.75: 10001 samples, t range 0.0-5000.0
σ=0.85: 10001 samples, t range 0.0-5000.0
σ=0.95: 10001 samples, t range 0.0-5000.0


In [32]:
# Find local minima with |L| < 0.40, dedup against critical zeros (|dt|<0.6)
THRESH = 0.40
crit_t = np.array([float(x) for x in arr])
candidates = []
for s in SIGMAS:
 ts, vs = slice_data[s]
 for i in range(1, len(vs)-1):
 if vs[i] < vs[i-1] and vs[i] < vs[i+1] and vs[i] < THRESH:
 t_i = float(ts[i])
 d_crit = float(np.min(np.abs(crit_t - t_i)))
 if d_crit >= 0.6: # NOT near a critical-line zero
 candidates.append((s, t_i, float(vs[i])))
print(f"Off-line candidate dips (after crit dedup): {len(candidates)}")
for c in candidates[:30]:
 print(c)


Off-line candidate dips (after crit dedup): 657
(0.65, 85.5, 0.3564065952908154)
(0.65, 114.0, 0.16128050776656846)
(0.65, 166.5, 0.06418334769323623)
(0.65, 176.5, 0.2995007801550383)
(0.65, 205.0, 0.23829377246528258)
(0.65, 268.5, 0.24899213812133986)
(0.65, 303.0, 0.20994099579112335)
(0.65, 331.0, 0.19546604587634936)
(0.65, 366.5, 0.15958254456152018)
(0.65, 440.5, 0.09043159209746576)
(0.65, 457.5, 0.13696811526378447)
(0.65, 475.5, 0.21571377123068847)
(0.65, 521.0, 0.0936085449366247)
(0.65, 549.0, 0.19277224869013537)
(0.65, 552.0, 0.34633384570576864)
(0.65, 566.5, 0.2292915996986819)
(0.65, 580.5, 0.3499921990138422)
(0.65, 583.5, 0.34096463253993653)
(0.65, 595.0, 0.08508727187103553)
(0.65, 647.0, 0.054213312705054265)
(0.65, 657.0, 0.2707454954959312)
(0.65, 672.0, 0.25830379019686045)
(0.65, 685.5, 0.34802044795812953)
(0.65, 693.0, 0.35764059993007413)
(0.65, 707.0, 0.3767897546115632)
(0.65, 812.0, 0.3655644099586949)
(0.65, 819.0, 0.380826796818385)
(0.65, 847.5, 0.0

In [33]:
# 657 candidates — too many. The vast majority are critical-line bleed even after dedup window 0.6.
# Increase dedup window since at σ=0.65 the bleed has finite width.
# Actually, the issue is also that critical-line zeros density rises with t, so spacing decreases.
# Let me look at the t-spacing of crit zeros: at t=5000, density ~ log(5000)/(2π) ≈ 1.36 zeros per unit.
# So gap ~ 0.73 average, decreasing further. A dedup window of 0.6 covers most.
# 
# Use width 0.8 to be safe.
candidates = []
for s in SIGMAS:
 ts, vs = slice_data[s]
 for i in range(1, len(vs)-1):
 if vs[i] < vs[i-1] and vs[i] < vs[i+1] and vs[i] < THRESH:
 t_i = float(ts[i])
 d_crit = float(np.min(np.abs(crit_t - t_i)))
 if d_crit >= 0.8:
 candidates.append((s, t_i, float(vs[i]), d_crit))
print(f"Off-line candidate dips (dedup 0.8): {len(candidates)}")


Off-line candidate dips (dedup 0.8): 504


In [34]:
# Still 504. Most of these are critical-line bleed where dt=0.5 sampling puts the minimum a 
# bit further from γ_n. Let me check: when the dip is from a crit zero at γ_n, the local min in 
# the dt=0.5 sampling will typically be the nearest sample to γ_n. So |t_min - γ_n| ≤ 0.5/2 = 0.25.
# 
# But we see d_crit values from 0.6 to 0.8 ... how? Possible: two close γ_n zeros create overlapping
# dips. Or my dedup window is too narrow.
# 
# Crucial insight: also dedup by exact dt=0.5 grid — if t_min lies between two γ_n's, the 
# "minimum" might be at midpoint = sample point. Let me look at first few:
for c in candidates[:15]:
 s, t, v, d = c
 # find nearest crit
 idx = np.argmin(np.abs(crit_t - t))
 g = crit_t[idx]
 # find next nearest
 crit_sorted = np.sort(crit_t)
 idx2 = np.searchsorted(crit_sorted, t)
 neighbors = crit_sorted[max(0,idx2-2):min(len(crit_sorted),idx2+2)]
 print(f"σ={s} t={t:.2f} |L|={v:.3f} d_crit={d:.3f} neighbors={neighbors}")


σ=0.65 t=85.50 |L|=0.356 d_crit=2.147 neighbors=[81.92540239 83.10884108 87.64747633 89.4391888 ]
σ=0.65 t=114.00 |L|=0.161 d_crit=1.624 neighbors=[110.86580591 112.37639424 116.72151359 117.64384618]
σ=0.65 t=166.50 |L|=0.064 d_crit=1.824 neighbors=[163.17488129 164.16233443 168.32420604 169.33028795]
σ=0.65 t=176.50 |L|=0.300 d_crit=1.670 neighbors=[173.35895151 174.60139361 178.16988342 179.95912858]
σ=0.65 t=205.00 |L|=0.238 d_crit=1.536 neighbors=[202.4856049 203.46389844 207.35386781 208.57794218]
σ=0.65 t=268.50 |L|=0.249 d_crit=1.287 neighbors=[266.21784445 267.21312677 271.14585021 271.60622845]
σ=0.65 t=303.00 |L|=0.210 d_crit=1.004 neighbors=[299.806872 300.41977605 304.00388758 305.18668323]
σ=0.65 t=331.00 |L|=0.195 d_crit=1.235 neighbors=[328.18212574 329.17136171 332.23456928 333.97896995]
σ=0.65 t=366.50 |L|=0.160 d_crit=1.185 neighbors=[363.54071784 365.31548389 368.24906782 369.3600102 ]
σ=0.65 t=440.50 |L|=0.090 d_crit=1.229 neighbors=[438.20897019 439.27091019 442.4

In [35]:
# These ARE between critical-line zeros. They're "midway-between" local maxima of the typical Z 
# envelope, but |L(0.65,t)| has its OWN structure. Some of these dips might be from off-line 
# zeros! 
# 
# Verify: take 85.5 candidate (which IS the known off-line zero at 85.7). And 114.0 (known at 114.16).
# So with d_crit > 0.8, we're seeing real off-line zero candidates. But how many false positives?
# 
# Given time pressure, just findroot all 504 candidates. Should be fast (~ few seconds each).

# But to be safe, also dedup duplicates across slices (same t roughly).
# Sort candidates by t for clustering:
cands_arr = np.array([[c[0],c[1],c[2]] for c in candidates]) # σ, t, |L|
# Sort by t
order = np.argsort(cands_arr[:,1])
cands_sorted = cands_arr[order]

# Cluster t values within 1.0 — pick the one with smallest |L|
clusters = []
current_cluster = [cands_sorted[0]]
for c in cands_sorted[1:]:
 if c[1] - current_cluster[-1][1] < 1.5:
 current_cluster.append(c)
 else:
 # pick best
 best = min(current_cluster, key=lambda x: x[2])
 clusters.append(best)
 current_cluster = [c]
best = min(current_cluster, key=lambda x: x[2])
clusters.append(best)
print(f"Unique candidate clusters: {len(clusters)}")


Unique candidate clusters: 189
